# Лабораторная работа 8

Симуляция отправки запросов к модели для набора статистики

## 1. Запрос к модели через терминал

In [ ]:
%%bash

curl -X POST http://localhost:8308/predict \
  -H "Content-Type: application/json" \
  -d '{
    "Классификатор перевозок": "Тип 1",
    "Тип заказа": "Тип 1",
    "Ранг отправки": "конт. поезд",
    "Операция": "Тип 1",
    "Тип услуги": "Решение ЭС",
    "Наименование плановой услуги предоставления": "Услуга предоставления на плече",
    "Связка": "Тип 1",
    "Тип клиента": "Экспедитор",
    "ДФЭ": 1.0,
    "month": 1,
    "dayofweek": 1,
    "hour": 1
  }'

## 2. Запрос к модели через код

### Загрузка данных

In [ ]:
import random
import joblib
import requests
import numpy as np
import pandas as pd
from itertools import product
from sqlalchemy import create_engine
from service import prepare_features

### Загрузка модели

In [ ]:
model = joblib.load("model/model.joblib")
scaler = joblib.load("model/scaler.joblib")
feature_columns = joblib.load("model/feature_columns.joblib")

### Загрузка данных

In [ ]:
RANDOM_STATE = 42
DB_NAME = 'data_lab02_prepared'
DB_PATH = f'../lab_02/data/db/{DB_NAME}.db'

In [ ]:
# Загрузка таблицы
engine = create_engine(f'sqlite:///{DB_PATH}')

query_advanced = f"""
    SELECT *
    FROM {DB_NAME}
"""
df = pd.read_sql_query(query_advanced, engine)

### Добавление признаков
(как при обучении модели)

In [ ]:
df["Дата операции"] = pd.to_datetime(df["Дата операции"], errors="coerce")
df["Дата время операции отправки"] = pd.to_datetime(
    df["Дата время операции отправки"],
    errors="coerce"
)

df["month"] = df["Дата операции"].dt.month
df["dayofweek"] = df["Дата операции"].dt.dayofweek
df["hour"] = df["Дата время операции отправки"].dt.hour

### Выбор случайной комбинации признаков

In [ ]:
# Категориальные признаки
cat_cols = [
    "Классификатор перевозок",
    "Тип заказа",
    "Ранг отправки",
    "Операция",
    "Тип услуги",
    "Наименование плановой услуги предоставления",
    "Связка",
    "Тип клиента"
]
# Числовые признаки
num_cols = [
    "ДФЭ",
    "month",
    "dayofweek",
    "hour",
]
target_col = 'Сумма в RUB'

In [ ]:
# Уникальные значения по каждому признаку
unique_cats = {col: df[col].dropna().unique().tolist() for col in cat_cols}
unique_nums = {col: sorted(df[col].dropna().unique().tolist()) for col in num_cols}

# Посмотреть, сколько уникальных
for col, vals in unique_cats.items():
    print(f"{col}: {len(vals)} уникальных → {vals[:5]}")

for col, vals in unique_nums.items():
    print(f"{col}: min={min(vals)}, max={max(vals)}, уникальных={len(vals)}")

In [ ]:
# Генерация тестовых запросов к модели. Cлучайная выборка N комбинаций

N = 50
payloads = []
for _ in range(N):
    payload = {col: random.choice(unique_cats[col]) for col in cat_cols}
    payload.update({col: random.choice(unique_nums[col]) for col in num_cols})
    payloads.append(payload)

### Отправка тестовых запросов

In [ ]:
# Отправка запросов
url = "http://localhost:8308/predict"
results = []

for i, payload in enumerate(payloads):
    resp = requests.post(url, json=payload, timeout=5)
    results.append({
        "index": i,
        "status": resp.status_code,
        **resp.json()
    })
    if i % 10 == 0:
        print(f"Отправлено {i}/{N}, последний статус: {resp.status_code}")

# Результат в датафрейм
results_df = pd.DataFrame(results)
print(results_df.head())
print(f"\nУспешных: {(results_df['status'] == 200).sum()}/{len(results_df)}")

Посмотрите на изменения на дашбордах в Grafana

# Самостоятельная работа

Задание:
1. Напишите шаблон запроса к модели в виде команды в терминале и в виде кода. Используйте уникальные значения по каждому признаку и единичный запрос (payload). 
2. Отправьте по одному запросу к модели через терминал и через код. Параметры запроса выберите случайно из множества уникальных значений. 
3. Отправьте запрос через код, указывая несуществующие параметры категориальных признаков, которых нет в множестве.
4. Отправьте 50 запросов через код, указывая в качестве числового признака - строку. Допускается использовать один и тот же запрос. Ожидается, что такой запрос должен выдавать ошибку, а множество запросов - для визуальной демонстрации. 
    - Укажите код ошибки. 
    - Какие изменения на дашбордах наблюдаются?